# European Flight Delay Prediction — Data Cleaning nad Preprocessing

This notebook prepares the training, development and test datasets created by '03_data_splitting.ipynb' by doing the following
* Load the existing datasets from GCS
* Validate the input data and target variable
* Select suitable predictors and exclude fields that could cause data leakage
* Handle missing values and encode categorical features
* Fit preprocessing using training data
* Apply the same transformations to development and test data
* Save the processed output sand record information needed for reproducibility


## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [ ]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [ ]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME
*   SPLIT_RUN_ID - this should reference existing notebook 03 split run


In [ ]:

from google.colab import auth
auth.authenticate_user()

from pathlib import Path
from google.cloud import storage
import pandas as pd
import json

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"

# Existing Notebook 03 split run
SPLIT_RUN_ID = "20261020T065554Z-cc2d29384141"
SPLIT_PREFIX = f"processed/splits/runs/{SPLIT_RUN_ID}"

TARGET_COLUMN = "delayed_15min"

WORK_DIR = Path("/content/flight_preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# Confirm the source run exists
provenance_blob = bucket.blob(
    f"{SPLIT_PREFIX}/provenance.json"
)

if not provenance_blob.exists():
    raise FileNotFoundError(
        f"Split run not found: gs://{BUCKET_NAME}/{SPLIT_PREFIX}"
    )

source_provenance = json.loads(
    provenance_blob.download_as_text()
)

assert source_provenance["run_id"] == SPLIT_RUN_ID

print("Connected to Google Cloud Storage.")
print("Source split run:", SPLIT_RUN_ID)
print("Source split sizes:", source_provenance["split_rows"])


## 3. Load and Validate the Existing Datasets

Load the training, development and test datasets created by Notebook 03.

Complete the following validation checks to help identify potential problems before preprocessing
- All three datasets are available in Google Cloud Storage
- The datasets contain the expected columns
- The prediction target contains only valid values (0 or 1)
- The datasets contain the expected number of records
- The target distribution is consistent across the three datasets


In [ ]:

#Load the 3 traning, development and test datasets

from io import BytesIO

def load_parquet_from_gcs(filename):
    """Load a Parquet file from the selected Notebook 03 run."""
    blob_path = f"{SPLIT_PREFIX}/{filename}"
    blob = bucket.blob(blob_path)

    if not blob.exists():
        raise FileNotFoundError(
            f"Missing input file: gs://{BUCKET_NAME}/{blob_path}"
        )

    return pd.read_parquet(BytesIO(blob.download_as_bytes()))


train_df = load_parquet_from_gcs("flights_train.parquet")
dev_df = load_parquet_from_gcs("flights_dev.parquet")
test_df = load_parquet_from_gcs("flights_test.parquet")

print("Datasets loaded successfully.")
print(f"Training:    {len(train_df):,} records")
print(f"Development: {len(dev_df):,} records")
print(f"Test:        {len(test_df):,} records")


In [ ]:
# Validate the datasets

datasets = {
    "Training": train_df,
    "Development": dev_df,
    "Test": test_df
}

expected_sizes = {
    "Training": 80000,
    "Development": 10000,
    "Test": 10000
}

# Check record counts
for name, df in datasets.items():
    assert len(df) == expected_sizes[name], (
        f"{name}: unexpected record count"
    )

# Check column consistency
expected_columns = set(train_df.columns)

for name, df in datasets.items():
    assert set(df.columns) == expected_columns, (
        f"{name}: columns differ from training data"
    )

    assert TARGET_COLUMN in df.columns, (
        f"{name}: target column missing"
    )

    assert df[TARGET_COLUMN].notna().all(), (
        f"{name}: missing target values"
    )

    assert df[TARGET_COLUMN].isin([0, 1]).all(), (
        f"{name}: invalid target values"
    )

# Check flight identifiers do not overlap between splits
if "id" in expected_columns:
    train_ids = set(train_df["id"])
    dev_ids = set(dev_df["id"])
    test_ids = set(test_df["id"])

    assert train_ids.isdisjoint(dev_ids), (
        "Training and development IDs overlap"
    )
    assert train_ids.isdisjoint(test_ids), (
        "Training and test IDs overlap"
    )
    assert dev_ids.isdisjoint(test_ids), (
        "Development and test IDs overlap"
    )

print("All input validation checks passed.")


In [ ]:
# Check the delayed flight percentages

summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Records": len(df),
        "Delayed Flights": int(df[TARGET_COLUMN].sum()),
        "Delay Rate (%)": round(
            df[TARGET_COLUMN].mean() * 100, 3
        )
    })

validation_summary = pd.DataFrame(summary)

display(validation_summary)


## 4. Feature Selection and Data Leakage Prevention

Select the input features that will be used to predict whether a flight is delayed by more than 15 minutes based on the exploratory data anlysis completed in Notebook 04.

We will exclude certain columns which contain information that would not be available at the time of prediction, those which could reveal the actual dealy along with those that are identifiers rather than useful predictors.

The same feature selection will be applied to the training, development and test datasets.

The target variable delayed_15min is kept separately from the input features.

In [ ]:

# Candidate input features based on Notebook 04 EDA
FEATURE_COLUMNS = [
    "adep",
    "ades",
    "route",
    "source_year",
    "source_month",
    "expected_duration_min",
    "typecode",
    "model",
    "icao_aircraft_class",
    "icao_operator"
]

DATE_COLUMN = "dof"

# Columns deliberately excluded to avoid leakage
LEAKAGE_COLUMNS = [
    "delay_min",
    "duration_min",
    "first_seen",
    "last_seen"
]

# Confirm all required columns are available
required_columns = FEATURE_COLUMNS + [
    DATE_COLUMN,
    TARGET_COLUMN
]

for name, df in datasets.items():
    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} dataset missing columns: {missing}"
        )

print("All selected feature columns are available.")
print(f"Selected original features: {len(FEATURE_COLUMNS) + 1}")


### 4.1 Extract useful information from the flight date

Instead of using full date we will extract the day of week where Monday is 0, Tuesday is 1 etc.

This could help the model identify whether delays are more common on particular days

In [ ]:

def prepare_features(df):
    """Select predictors and create a day-of-week feature."""

    X = df[FEATURE_COLUMNS].copy()

    flight_dates = pd.to_datetime(
        df[DATE_COLUMN],
        errors="coerce"
    )

    if flight_dates.isna().any():
        raise ValueError(
            "Invalid or missing flight dates found."
        )

    X["day_of_week"] = flight_dates.dt.dayofweek

    y = df[TARGET_COLUMN].astype(int).copy()

    return X, y


X_train, y_train = prepare_features(train_df)
X_dev, y_dev = prepare_features(dev_df)
X_test, y_test = prepare_features(test_df)

print("Feature selection completed.")
print("Training features:", X_train.shape)
print("Development features:", X_dev.shape)
print("Test features:", X_test.shape)


### 4.2 Confirm no leakage columns remain




In [ ]:

# Confirm prohibited columns have not been selected
prohibited_columns = set(
    LEAKAGE_COLUMNS + [
        TARGET_COLUMN,
        "id",
        "flt_id",
        "icao24",
        "registration"
    ]
)

for name, X in [
    ("Training", X_train),
    ("Development", X_dev),
    ("Test", X_test)
]:
    assert not prohibited_columns.intersection(X.columns), (
        f"{name}: prohibited columns detected"
    )

    assert list(X.columns) == list(X_train.columns), (
        f"{name}: feature columns do not match training"
    )

print("Feature selection validation passed.")
print("No prohibited or leakage columns selected.")

display(X_train.head())


## 5. Handling Missing Values and Categorical Encoding

Create a reproducible preprocessing pipeline using scikit-learn based on the data quality findings from Notebook 04
* Missing categorical values are replaced with 'Unknown'
* Missing numerical values are replaced with the median calculated from training data
* Categorical values are converted into numerical features using one-hot encoding
* Infrequent categories are grouped to reduce the number of output columns
* Numerical features are scaled to a comparable range

The preprocessing pipeline is fitted using training data only.

The development and test datasets are transformed using the same fitted pipeline to prevent data leakage


### 5.1 Define Categorical and Numerical Features

In [ ]:

# Separate categorical and numerical features

CATEGORICAL_FEATURES = [
    "adep",
    "ades",
    "route",
    "typecode",
    "model",
    "icao_aircraft_class",
    "icao_operator"
]

NUMERICAL_FEATURES = [
    "source_year",
    "source_month",
    "expected_duration_min",
    "day_of_week"
]

# Confirm all selected features are accounted for
assert set(CATEGORICAL_FEATURES + NUMERICAL_FEATURES) == set(
    X_train.columns
), "Feature lists do not match the selected training features."

print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Numerical features:", len(NUMERICAL_FEATURES))
print("All features accounted for.")


### 5.2 Create Preprocessing Pipeline

Use OneHotEncoder wtih a minimum frequency threshold of 100 i.e. a category must appear at least 100 times to receive its own encoded column.

Note: route has over 18,000 distinct values, rather than creating a separate column for each route the encoder will group infrequent cateegories together

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=100,
            sparse_output=True
        )
    )
])

# Combine both preprocessing approaches
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, NUMERICAL_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES)
    ],
    sparse_threshold=1.0
)

print("Preprocessing pipeline created successfully.")


### 5.3 Fit using Training Data


In [ ]:

# Learn preprocessing rules from training data ONLY
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the SAME rules to development and test
X_dev_processed = preprocessor.transform(X_dev)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed.")
print("Training:", X_train_processed.shape)
print("Development:", X_dev_processed.shape)
print("Test:", X_test_processed.shape)


### 5.4 Validate the processed outputs

In [ ]:

import numpy as np
from scipy import sparse

# Check matching feature dimensions
assert (
    X_train_processed.shape[1]
    == X_dev_processed.shape[1]
    == X_test_processed.shape[1]
), "Processed feature counts do not match."

# Check record counts remain unchanged
assert X_train_processed.shape[0] == len(X_train)
assert X_dev_processed.shape[0] == len(X_dev)
assert X_test_processed.shape[0] == len(X_test)

# Check no missing or infinite values remain
for name, matrix in [
    ("Training", X_train_processed),
    ("Development", X_dev_processed),
    ("Test", X_test_processed)
]:
    values = matrix.data if sparse.issparse(matrix) else matrix

    assert np.isfinite(values).all(), (
        f"{name}: missing or infinite values found"
    )

print("All preprocessing validation checks passed.")
print("Final feature count:", X_train_processed.shape[1])


## 6. Save Preprocessed Data and Reproducibility Information

Save the outputs to Google Cloud storage including

*   Processed training, development and test matrices
*   The fitted processing pipeline
*   The selected feature names
*.  A provenance file recording the source split run, configuration and output checksums



### 6.1 Create unique output folder

In [ ]:

from datetime import datetime, timezone
import subprocess

def get_git_commit():
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()

git_commit = get_git_commit()

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

PREPROCESS_RUN_ID = f"{timestamp}-{git_commit[:12]}"

OUTPUT_PREFIX = (
    f"processed/preprocessing/runs/{PREPROCESS_RUN_ID}"
)

print("Preprocessing run:", PREPROCESS_RUN_ID)
print("GCS output location:")
print(f"gs://{BUCKET_NAME}/{OUTPUT_PREFIX}/")


### 6.2 Save Processed Datasets Locally

Storing in sparce .npz format as one-hot encoding creates a sparse matrix with mostly zero's. This format allows us to store the data efficiently without wasting space saving all those zerios in a large table.

The npy files contain only the target variable delayed_15min for each flight

In [ ]:

from scipy import sparse
import numpy as np
import joblib

OUTPUT_DIR = WORK_DIR / PREPROCESS_RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

feature_matrices = {
    "train": X_train_processed,
    "dev": X_dev_processed,
    "test": X_test_processed
}

target_arrays = {
    "train": y_train,
    "dev": y_dev,
    "test": y_test
}

for split_name, matrix in feature_matrices.items():
    sparse.save_npz(
        OUTPUT_DIR / f"X_{split_name}.npz",
        sparse.csr_matrix(matrix)
    )

    np.save(
        OUTPUT_DIR / f"y_{split_name}.npy",
        target_arrays[split_name].to_numpy()
    )

# Save fitted preprocessing rules for reuse
joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.joblib"
)

# Save the names and order of the 501 output features
feature_names = preprocessor.get_feature_names_out().tolist()

with open(
    OUTPUT_DIR / "feature_names.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(feature_names, f, indent=2)

print("Files saved locally.")
print("Output features:", len(feature_names))
print("Output folder:", OUTPUT_DIR)


### 6.3 Create Provenance File

In [ ]:

import hashlib
import platform
import sklearn
import scipy

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


provenance = {
    "run_id": PREPROCESS_RUN_ID,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "git_commit": git_commit,
    "source_split_run": SPLIT_RUN_ID,
    "source_gcs_prefix": (
        f"gs://{BUCKET_NAME}/{SPLIT_PREFIX}"
    ),
    "target_column": TARGET_COLUMN,
    "input_features": list(X_train.columns),
    "output_feature_count": len(feature_names),
    "rows": {
        "train": len(X_train),
        "dev": len(X_dev),
        "test": len(X_test)
    },
    "preprocessing": {
        "categorical_imputation": "constant: Unknown",
        "numerical_imputation": "median",
        "categorical_encoding": "one-hot",
        "minimum_category_frequency": 100,
        "unknown_category_handling": "infrequent_if_exist",
        "numerical_scaling": "StandardScaler",
        "fitted_on": "training only"
    },
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__
    }
}

# Record checksums for all saved output files
provenance["output_sha256"] = {
    path.name: sha256_file(path)
    for path in sorted(OUTPUT_DIR.iterdir())
    if path.is_file()
}

with open(
    OUTPUT_DIR / "provenance.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(provenance, f, indent=2)

print("Provenance file created.")
print("Files recorded:", len(provenance["output_sha256"]))


### 6.4 Upload outputs to Google Cloud Storage

In [ ]:

uploaded_files = []

for local_path in sorted(OUTPUT_DIR.iterdir()):
    if not local_path.is_file():
        continue

    object_name = f"{OUTPUT_PREFIX}/{local_path.name}"
    blob = bucket.blob(object_name)

    # Prevent accidental overwriting of existing files
    blob.upload_from_filename(
        str(local_path),
        if_generation_match=0
    )

    blob.reload()

    uploaded_files.append({
        "file": local_path.name,
        "gcs_generation": blob.generation
    })

    print(f"Uploaded: {local_path.name}")

print()
print("Upload complete.")
print("Files uploaded:", len(uploaded_files))
print(f"gs://{BUCKET_NAME}/{OUTPUT_PREFIX}/")


### 6.5 Verify Uploaded files

In [ ]:

for local_path in sorted(OUTPUT_DIR.iterdir()):
    if not local_path.is_file():
        continue

    object_name = f"{OUTPUT_PREFIX}/{local_path.name}"
    blob = bucket.blob(object_name)

    assert blob.exists(), (
        f"Missing GCS file: {object_name}"
    )

    # Compare SHA-256 checksums
    uploaded_hash = hashlib.sha256(
        blob.download_as_bytes()
    ).hexdigest()

    local_hash = sha256_file(local_path)

    assert uploaded_hash == local_hash, (
        f"Checksum mismatch: {local_path.name}"
    )

print("All uploaded files verified successfully.")
print("Preprocessing run:", PREPROCESS_RUN_ID)
print("Final feature count:", len(feature_names))


## 7. Reproducability Verification

Verify that the saved preprocessing outputs can be retrieved and reproduced by

* Downloaded the fitted preprocessing pipeline from GCS
* Reload the saved training, development and test feature matrices
* Apply the reloaded pipeline to the original input features
* Compare the reproduced results against the saved outpus
* Verify that the target values and feature names remain unchanged



### 7.1 Download and reload the saved files

In [ ]:

from scipy import sparse
import joblib
import numpy as np
import json

# Create a separate folder for downloaded verification files
VERIFY_DIR = WORK_DIR / "reproducibility_check"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)

files_to_verify = [
    "X_train.npz",
    "X_dev.npz",
    "X_test.npz",
    "y_train.npy",
    "y_dev.npy",
    "y_test.npy",
    "preprocessor.joblib",
    "feature_names.json",
    "provenance.json"
]

# Download the files from GCS
for filename in files_to_verify:
    blob = bucket.blob(f"{OUTPUT_PREFIX}/{filename}")
    blob.download_to_filename(str(VERIFY_DIR / filename))

# Reload fitted preprocessing pipeline
reloaded_preprocessor = joblib.load(
    VERIFY_DIR / "preprocessor.joblib"
)

# Reload saved feature matrices
saved_matrices = {
    "train": sparse.load_npz(VERIFY_DIR / "X_train.npz"),
    "dev": sparse.load_npz(VERIFY_DIR / "X_dev.npz"),
    "test": sparse.load_npz(VERIFY_DIR / "X_test.npz")
}

# Reload target values
saved_targets = {
    "train": np.load(VERIFY_DIR / "y_train.npy"),
    "dev": np.load(VERIFY_DIR / "y_dev.npy"),
    "test": np.load(VERIFY_DIR / "y_test.npy")
}

with open(VERIFY_DIR / "feature_names.json") as f:
    saved_feature_names = json.load(f)

with open(VERIFY_DIR / "provenance.json") as f:
    saved_provenance = json.load(f)

print("All files downloaded and reloaded successfully.")
print("Saved feature count:", len(saved_feature_names))


### 7.2 Reproduced the Processed Datasets

Note: Using transofrm here rather than fit_transform as want to reuse the existing preprocessing rules

In [ ]:

# Reproduce processed features using the reloaded pipeline

original_features = {
    "train": X_train,
    "dev": X_dev,
    "test": X_test
}

original_targets = {
    "train": y_train,
    "dev": y_dev,
    "test": y_test
}

for split_name in ["train", "dev", "test"]:

    reproduced = reloaded_preprocessor.transform(
        original_features[split_name]
    )

    saved = saved_matrices[split_name]

    # Compare matrix dimensions
    assert reproduced.shape == saved.shape, (
        f"{split_name}: shape mismatch"
    )

    # Compare all numerical feature values
    difference = (
        sparse.csr_matrix(reproduced) - saved
    )

    assert np.allclose(
        difference.data,
        0,
        atol=1e-10
    ), f"{split_name}: feature values differ"

    # Compare target values
    assert np.array_equal(
        original_targets[split_name].to_numpy(),
        saved_targets[split_name]
    ), f"{split_name}: target values differ"

    print(
        f"{split_name.upper()}: "
        f"PASS - {saved.shape[0]:,} rows, "
        f"{saved.shape[1]} features"
    )

print()
print("All three datasets reproduced successfully.")


### 7.3 Verify feature names and provenance

In [ ]:

# Verify feature names and order
reloaded_feature_names = (
    reloaded_preprocessor.get_feature_names_out().tolist()
)

assert reloaded_feature_names == saved_feature_names, (
    "Saved feature names do not match the pipeline"
)

# Verify source and output metadata
assert saved_provenance["run_id"] == PREPROCESS_RUN_ID

assert saved_provenance["source_split_run"] == SPLIT_RUN_ID

assert saved_provenance["output_feature_count"] == 501

assert saved_provenance["rows"] == {
    "train": 80000,
    "dev": 10000,
    "test": 10000
}

# Verify downloaded files against recorded checksums
for filename, expected_hash in saved_provenance[
    "output_sha256"
].items():

    actual_hash = sha256_file(VERIFY_DIR / filename)

    assert actual_hash == expected_hash, (
        f"Checksum mismatch: {filename}"
    )

print("Feature names verified.")
print("Provenance metadata verified.")
print("File checksums verified.")
print()
print("REPRODUCIBILITY TEST: PASSED")
